# 07 — Sparse Autoencoders: Reading and Steering Features

Individual neurons are often *polysemantic*: one neuron fires for several
unrelated things, because the model packs more features than it has
dimensions (*superposition*). A **sparse autoencoder (SAE)** learns an
overcomplete dictionary of directions in which each activation is a sparse
combination — and those directions tend to be far more interpretable.

This notebook walks the full workflow on a tiny model:

1. collect activations and train an SAE,
2. read what features detect (top tokens and in-context examples),
3. see how features relate (co-occurrence and geometry),
4. test features causally (steering, steered generation, ablation),
5. save the dictionary for later.

In [ ]:
import sys
sys.path.insert(0, "..")  # run from the notebooks/ directory

import matplotlib
import torch

import kamui
from kamui.model.config import ModelConfig

torch.manual_seed(0)
print("KAMUI", kamui.__version__)

## Setup: a tiny model trained on structured text

An SAE can only find features the model actually computes. So instead of a
random word salad (where the best model just memorises word frequencies and
its activations barely change from token to token), we train on templated
sentences — *the ANIMAL VERB PREP the OBJECT .* — whose next word depends on
context. (KAMUI's BPE merges frequent phrases such as `"sat on the "` into one
token; that is fine here.)

In [ ]:
import random

from kamui.tokenizer.bpe import BPETokenizer
from kamui.training import DataLoader, TextDataset, Trainer, TrainingConfig

rng = random.Random(0)
ANIMALS, VERBS = ["cat", "dog", "bird", "fish"], ["sat", "ran", "slept", "hid"]
PREPS, OBJECTS = ["on", "under", "near"], ["mat", "log", "rug", "box"]
CORPUS = " ".join(
    f"the {rng.choice(ANIMALS)} {rng.choice(VERBS)} {rng.choice(PREPS)} the {rng.choice(OBJECTS)} ."
    for _ in range(700)
)

config = ModelConfig(n_layers=2, d_model=64, n_heads=4, d_ff=128,
                     vocab_size=320, context_length=32, dropout=0.0)
tokenizer = BPETokenizer.train(CORPUS, vocab_size=config.vocab_size)
tokens = tokenizer.encode(CORPUS)

model = kamui.KAMUITransformer(config)
trainer = Trainer(
    model,
    DataLoader(TextDataset(tokens, config.context_length), batch_size=8, seed=0),
    config=TrainingConfig(max_lr=3e-3, warmup_steps=10, max_steps=1000),
)
records = trainer.train(200)
model.eval()
print(f"loss: {records[0]['train_loss']:.3f} -> {records[-1]['train_loss']:.3f}")


def show(token_id):
    """Readable text for one token id (raw byte tokens may not be valid UTF-8 alone)."""
    try:
        return repr(tokenizer.decode([token_id]))
    except ValueError:
        return f"<byte {token_id}>"

## 1. Collect activations and train an SAE

We cache the output of the second block's feed-forward layer for every token
of 40 corpus chunks (one row per token), then train an SAE with 4× more
features than dimensions. The L1 penalty pushes most features to zero on any
given token. Its strength has to be tuned to the activations' scale: here
`l1_coeff=1.0` keeps roughly 8 of 256 features active per token while still
reconstructing almost all of the variance.

In [ ]:
from kamui.mechinterp import (
    SparseAutoencoder, collect_activations, sae_feature_metrics, train_sae,
)

HOOK = "blocks.1.ffn.output"
seqs = [torch.tensor(tokens[i : i + config.context_length])
        for i in range(0, 40 * config.context_length, config.context_length)]
acts = collect_activations(model, HOOK, seqs)          # (tokens, d_model)
token_ids = torch.cat(seqs)                            # one id per activation row
print("activations:", tuple(acts.shape))

sae = SparseAutoencoder(d_model=config.d_model, n_features=4 * config.d_model, l1_coeff=1.0)
train_sae(sae, acts, epochs=60, lr=3e-3, batch_size=256)
metrics = sae_feature_metrics(sae, acts)
explained = 1 - metrics.reconstruction_mse / acts.var(dim=0).mean().item()
print(f"variance explained: {explained:.1%}")
print(f"active features per token (L0): {metrics.mean_l0:.1f} of {sae.n_features}")
print(f"dead features: {metrics.dead_feature_fraction:.0%}")

## 2. What does each feature detect?

`interpret_features` lists the tokens that most strongly activate each
feature; `max_activating_examples` shows those peaks *in context*. Look for
features that fire on one *kind* of word — animals, verb phrases, sentence
endings.

In [ ]:
from kamui.mechinterp import interpret_features, max_activating_examples

profiles = interpret_features(sae, acts, token_ids, top_k=8)
# Keep features that fire on a subset of tokens, strongest first.
selective = [p for p in profiles if p.top_token_ids and p.density <= 0.5]
chosen = sorted(selective, key=lambda p: p.mean_activation, reverse=True)[:5]

for p in chosen:
    tops = sorted({show(i) for i in p.top_token_ids})
    print(f"feature {p.feature:3d}  fires on {p.density:4.0%} of tokens  top tokens: {', '.join(tops)}")

In [ ]:
feature = chosen[0].feature
for ex in max_activating_examples(model, sae, HOOK, seqs, feature, top_k=4, window=3):
    words = [tokenizer.decode([i]) for i in ex.context_token_ids]
    words[ex.focus_offset] = f"[{words[ex.focus_offset]}]"   # mark the peak token
    print(f"{ex.activation:5.2f}  {''.join(words)}")

## 3. How do features relate?

Two complementary views: **co-occurrence** asks whether features fire on the
same tokens; **similarity** asks whether their decoder directions point the
same way (near-parallel features are a sign of feature splitting).

In [ ]:
from kamui.mechinterp import feature_cooccurrence, feature_similarity
from kamui.utils.plotting import heatmap

ids_ = [p.feature for p in chosen]
labels = [f"f{i}" for i in ids_]
heatmap(feature_cooccurrence(sae, acts, features=ids_), labels, labels,
        title="co-activation density")
heatmap(feature_similarity(sae, features=ids_), labels, labels,
        title="decoder cosine similarity")

## 4. Causal tests: steer, generate, ablate

Reading a feature is a hypothesis; intervening tests it. **Steering** adds a
feature's direction to the residual stream; **ablation** removes the feature's
contribution and checks whether the prediction changes.

Compare the promoted tokens below with the feature's top tokens above: what
*activates* a feature and what it *causes* downstream are different questions,
which is exactly why causal tests matter.

In [ ]:
from kamui.evaluate import generate
from kamui.mechinterp import FeatureSteerer

steerer = FeatureSteerer(model, sae)
prompt = "the cat sat on the "
ids = torch.tensor(tokenizer.encode(prompt))

result = steerer.steer_with_feature(ids, HOOK, feature=feature, coefficient=8.0)
print("most promoted tokens:", [(show(t), round(d, 2)) for t, d in result.top_promoted(5)])

plain = generate(model, tokenizer, prompt, max_new_tokens=8, use_cache=True)
steered = steerer.generate_steered_with_feature(
    tokenizer, prompt, HOOK, feature=feature, coefficient=8.0,
    max_new_tokens=8, use_cache=True,
)
print("plain:  ", plain)
print("steered:", steered)

In [ ]:
effect = steerer.ablate_features(ids, HOOK, features=[p.feature for p in chosen])
print(f"ablating the chosen features moves the next-token logits by up to "
      f"{effect.logit_delta.abs().max():.3f}")

### A steering direction from contrasting examples (no SAE needed)

`build_steering_vector` takes the difference of mean activations between two
sets of inputs. In this corpus a verb phrase (`"sat on the "`) is always
followed by an **object**, and a sentence ending (`"mat . the "`) by an
**animal** — so their difference is an *"an object comes next"* direction.
Adding it at a point where an animal should come next tests whether the model
really uses that direction.

In [ ]:
from kamui.mechinterp import build_steering_vector

words = {t: tokenizer.decode([t]) for t in set(tokens)}
first = {t: (w.split() or [""])[0] for t, w in words.items()}

verb_phrases = [t for t, w in words.items() if first[t] in VERBS and w.endswith("the ")]
endings = [t for t, w in words.items() if first[t] in OBJECTS and w.endswith(". the ")]
direction = build_steering_vector(
    model, HOOK,
    positive=[torch.tensor([t]) for t in verb_phrases],   # an object comes next
    negative=[torch.tensor([t]) for t in endings],        # an animal comes next
)

animals = [t for t in words if first[t] in ANIMALS]
objects = [t for t in words if first[t] in OBJECTS]
prompt2 = "the bird hid near the box . the "              # an animal should come next
slot = torch.tensor(tokenizer.encode(prompt2))
with torch.no_grad():
    before = model(slot.unsqueeze(0))[0, -1].softmax(-1)
after = steerer.steer(slot, HOOK, direction, coefficient=4.0).steered_logits[0, -1].softmax(-1)
print(f"P(animal next): {before[animals].sum():.2f} -> {after[animals].sum():.2f}")
print(f"P(object next): {before[objects].sum():.2f} -> {after[objects].sum():.2f}")
print("steered text:", steerer.generate_steered(
    tokenizer, prompt2, HOOK, direction, coefficient=4.0, max_new_tokens=6, use_cache=True,
))

## 5. Save the dictionary

Training an SAE costs compute; `save` / `load` keep it (config included).

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "sae.pt"
    sae.save(path)
    restored = SparseAutoencoder.load(path)
print("restored encodes identically:", torch.equal(restored.encode(acts), sae.encode(acts)))

## Where next

- Train SAEs on other hook points (`blocks.0.ffn.output`, `embed.output`) and
  compare what they find.
- Sweep `l1_coeff`: more sparsity tends to mean cleaner but fewer live features.
- Combine with notebook 04: patch or ablate features instead of whole layers.